# 🩺 AI for Diabetes: Prediction, Monitoring & Classification
### A Proof of Concept for Clinical Decision Support

**Dataset:** Pima Indians Diabetes Database (NIDDK)  
**Goal:** Predict whether a patient has diabetes based on diagnostic measurements.  
**Audience:** Medical professionals exploring AI-assisted diagnostics.

---

## 1. Setup & Data Loading

In [ ]:
# === Install & Import ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, RocCurveDisplay
)
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.2)
print('✅ All libraries loaded.')

In [ ]:
# === Load the Pima Indians Diabetes Dataset ===
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]
df = pd.read_csv(url, names=columns)
print(f"Dataset shape: {df.shape}")
df.head(10)

### Clinical Features at a Glance
| Feature | Description | Unit |
|---------|-------------|------|
| Pregnancies | Number of pregnancies | count |
| Glucose | Plasma glucose (2h OGTT) | mg/dL |
| BloodPressure | Diastolic blood pressure | mm Hg |
| SkinThickness | Triceps skinfold thickness | mm |
| Insulin | 2-hour serum insulin | μU/mL |
| BMI | Body mass index | kg/m² |
| DiabetesPedigreeFunction | Hereditary diabetes risk score | — |
| Age | Age | years |
| **Outcome** | **0 = No diabetes, 1 = Diabetes** | — |

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# === Basic statistics ===
df.describe().T.style.background_gradient(cmap='YlOrRd', axis=1)

In [ ]:
# === Target Distribution ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['Outcome'].value_counts()
axes[0].pie(counts, labels=['No Diabetes (0)', 'Diabetes (1)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.05))
axes[0].set_title('Class Distribution')

# Count plot
sns.countplot(data=df, x='Outcome', ax=axes[1],
              palette={0: '#2ecc71', 1: '#e74c3c'})
axes[1].set_xticklabels(['No Diabetes', 'Diabetes'])
axes[1].set_title(f'Samples: {counts[0]} vs {counts[1]}')
plt.tight_layout()
plt.show()

print(f"⚠️  Class imbalance ratio: 1:{counts[0]/counts[1]:.1f}")

In [ ]:
# === Feature Distributions by Outcome ===
features = columns[:-1]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    for outcome, color, label in [(0, '#2ecc71', 'Healthy'), (1, '#e74c3c', 'Diabetic')]:
        subset = df[df['Outcome'] == outcome][feat]
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions: Healthy vs Diabetic', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === Correlation Heatmap ===
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔑 Top correlations with Outcome:")
print(corr['Outcome'].drop('Outcome').sort_values(ascending=False).to_string())

In [ ]:
# === Clinical Risk Zones (key scatter) ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [('Glucose', 'BMI'), ('Age', 'Glucose'), ('Insulin', 'Glucose')]

for ax, (x, y) in zip(axes, pairs):
    for outcome, color, label in [(0, '#2ecc71', 'Healthy'), (1, '#e74c3c', 'Diabetic')]:
        sub = df[df['Outcome'] == outcome]
        ax.scatter(sub[x], sub[y], c=color, label=label, alpha=0.5, edgecolors='w', s=40)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend()
    ax.set_title(f'{y} vs {x}')

plt.suptitle('Clinical Risk Scatter Plots', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing
Several features have **0 values that are biologically impossible** (e.g., Glucose=0, BMI=0). We treat these as missing and impute them.

In [ ]:
# === Detect & handle impossible zeros ===
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Biologically impossible zeros (→ treated as missing):")
for col in zero_cols:
    n_zeros = (df[col] == 0).sum()
    pct = n_zeros / len(df) * 100
    print(f"  {col:25s}: {n_zeros:3d} ({pct:.1f}%)")

# Replace zeros with NaN
df_clean = df.copy()
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)

print(f"\nTotal missing values after cleaning: {df_clean.isnull().sum().sum()}")

In [ ]:
# === KNN Imputation (clinically aware — uses patient similarity) ===
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print("✅ KNN imputation complete. Missing values remaining:", X_imputed.isnull().sum().sum())
X_imputed.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# === Train/Test Split + Scaling ===
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print(f"Train: {X_train_s.shape[0]} samples | Test: {X_test_s.shape[0]} samples")
print(f"Train positive rate: {y_train.mean():.1%} | Test positive rate: {y_test.mean():.1%}")

## 4. Predictive Modelling
We compare 4 models that are common in clinical ML studies.

In [ ]:
# === Define & train models ===
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='roc_auc')
    # Fit on full train
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'cv_auc_mean': cv_scores.mean(), 'cv_auc_std': cv_scores.std()
    }
    print(f"{name:25s} | CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# === ROC Curves ===
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curves — Diabetes Prediction')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# === Precision-Recall Curves (critical for imbalanced clinical data) ===
fig, ax = plt.subplots(figsize=(8, 7))

for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    ax.plot(rec, prec, color=color, lw=2, label=f"{name} (AP={ap:.3f})")

ax.axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label=f'Baseline ({y_test.mean():.2f})')
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision (Positive Predictive Value)')
ax.set_title('Precision-Recall Curves — Clinical Relevance')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# === Best Model — Detailed Report ===
best_name = max(results, key=lambda k: results[k]['cv_auc_mean'])
best = results[best_name]

print(f"🏆 Best model: {best_name} (CV AUC = {best['cv_auc_mean']:.3f})\n")
print(classification_report(y_test, best['y_pred'],
                            target_names=['Healthy', 'Diabetic']))

In [ ]:
# === Confusion Matrix ===
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Healthy', 'Diabetic'],
            yticklabels=['Healthy', 'Diabetic'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Clinical Summary:")
print(f"   True Positives  (correctly detected diabetic):  {tp}")
print(f"   True Negatives  (correctly cleared healthy):    {tn}")
print(f"   False Negatives (missed diabetic — DANGEROUS):  {fn}")
print(f"   False Positives (false alarm):                  {fp}")
print(f"   Sensitivity (recall): {tp/(tp+fn):.1%}")
print(f"   Specificity:          {tn/(tn+fp):.1%}")

## 5. Feature Importance — What Drives the Prediction?

In [ ]:
# === Feature importance (Random Forest) ===
rf = results['Random Forest']['model']
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot.barh(ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Feature Importance (Random Forest)', fontsize=14)
ax.set_xlabel('Gini Importance')
plt.tight_layout()
plt.show()

print("\n🔬 Top 3 predictors:")
for feat, val in importance.iloc[-3:].iloc[::-1].items():
    print(f"   {feat}: {val:.3f}")

## 6. Patient-Level Prediction Demo
Simulate how this model could assist a clinician.

In [ ]:
# === Simulate a new patient ===
new_patient = pd.DataFrame([{
    'Pregnancies': 2,
    'Glucose': 148,       # elevated (OGTT)
    'BloodPressure': 78,
    'SkinThickness': 30,
    'Insulin': 150,
    'BMI': 33.6,          # obese class I
    'DiabetesPedigreeFunction': 0.627,  # high family risk
    'Age': 50
}])

new_patient_s = scaler.transform(new_patient)

print("═" * 50)
print("  🩺 AI CLINICAL DECISION SUPPORT")
print("═" * 50)
print("\nPatient profile:")
for col in new_patient.columns:
    print(f"   {col:30s}: {new_patient[col].values[0]}")

print("\n--- Model Predictions ---")
for name, res in results.items():
    prob = res['model'].predict_proba(new_patient_s)[0][1]
    risk = '🔴 HIGH' if prob > 0.5 else '🟡 MODERATE' if prob > 0.3 else '🟢 LOW'
    print(f"   {name:25s}: {prob:.1%} probability → {risk}")

print("\n⚠️  This is a decision SUPPORT tool, not a diagnosis.")
print("   Final clinical judgment remains with the physician.")

## 7. Key Takeaways for Clinicians

| Aspect | Finding |
|--------|--------|
| **Best predictor** | Glucose level (plasma, 2h OGTT) |
| **Model performance** | AUC ~0.83–0.86 (competitive with published literature) |
| **Clinical utility** | High sensitivity = fewer missed cases |
| **Limitations** | Single-population dataset, no longitudinal data |
| **Next steps** | Validate on local hospital data, add HbA1c, fasting glucose |

### ⚖️ Ethical Considerations
- **Bias:** Trained only on Pima Indian women → not generalizable as-is  
- **Explainability:** Feature importance provides transparency  
- **Role:** AI augments, never replaces, clinical judgment  
- **Privacy:** Patient data must be de-identified and GDPR-compliant